# 06 — Process local drone HDF5 data

This is the cleaned, portable counterpart to the active `Drone_processing.ipynb`. It uses the same public `run_drone_pipeline` orchestrator and the same processed/failed/merged/QA checks, while keeping machine-specific transfer commands and saved outputs out of the reusable vignette.

## 1. Setup

Place local drone HDF5 files beneath one input directory; nested folders are discovered recursively. From a fresh clone, install with `python -m pip install -e ".[notebooks]"` and start Jupyter from the repository root. Data-transfer credentials and commands stay outside this notebook.

In [ ]:
from pathlib import Path
from pprint import pprint

import pandas as pd

from spectralbridge import run_drone_pipeline

RUN = False
input_h5_dir = Path("data/drone_h5")
polygon_path = Path("data/aop_macrosystems_data_1_7_25.geojson")
output_dir = Path("outputs/drone_notebook")
drone_manifest_path = None

## 2. Check inputs before processing

This makes path mistakes visible before a long run. The pipeline performs its own authoritative discovery and records the resolved input path in the QA summary.

In [ ]:
discovered_h5 = sorted(input_h5_dir.rglob("*.h5")) if input_h5_dir.exists() else []
print(f"Input directory: {input_h5_dir.resolve()}")
print(f"Input exists: {input_h5_dir.exists()} | HDF5 files found: {len(discovered_h5)}")
print(f"Polygon file: {polygon_path} | exists: {polygon_path.exists()}")
print(f"Output directory: {output_dir.resolve()}")
for path in discovered_h5[:10]:
    print(f"  {path}")

## 3. Run or resume

The returned dictionary is the first diagnostic: it separates processed and failed flights and points to the merged table and QA summary. Rerunning with `overwrite=False` preserves restart-safe behavior.

In [ ]:
results = None
if RUN:
    results = run_drone_pipeline(
        input_h5_dir=input_h5_dir,
        polygon_path=polygon_path,
        output_dir=output_dir,
        apply_topo=True,
        apply_brdf=True,
        overwrite=False,
        drone_manifest_path=drone_manifest_path,
        require_solar_geometry=True,
    )
    print(f"Processed: {len(results['processed'])}")
    print(f"Failed: {len(results['failed'])}")
    print(f"Merged parquet: {results['merged']}")
    print(f"QA summary: {results['qa_summary_path']}")
    pprint(results["qa_summary"])
else:
    print("Dry run. Check the paths above, then set RUN = True.")

## 4. Preview the merged table

As in the active drone notebook, open only the merged result after checking that the orchestrator returned a path. The preview confirms identifiers, coordinates, and reflectance columns are present.

In [ ]:
merged_path = Path(results["merged"]) if results and results.get("merged") else None
if RUN and merged_path is not None and merged_path.exists():
    merged_preview = pd.read_parquet(merged_path).head()
    display(merged_preview)
    print(f"Preview columns ({len(merged_preview.columns)}): {list(merged_preview.columns)}")
elif RUN:
    print("No merged Parquet was produced; inspect results['failed'] and the QA summary.")
else:
    print("The merged-table preview runs after the pipeline.")

## 5. What success looks like

Review failures even when some flights succeed. Confirm the QA summary reports the intended geometry and correction settings, then verify the merged table contains the expected flight identifiers and polygon records. Drone outputs remain separate from the NEON download workflow.